# Retail Demand Forecasting: Dataset to Model Evaluation (70/30)

This notebook loads the M5 sales, calendar, and price files; creates a chronological 70% training / 30% test split; selects boosting rounds using validation dates inside the training partition; refits on the full training period; recursively forecasts the held-out period in 28-day windows; compares two baselines; and saves the model, metrics, and run manifest.

It uses the repository's feature engineering and evaluation helpers in `src/forecast_experiments.py` and `src/forecast_holdout.py`. Raw data and generated artifacts remain local.

In [ ]:
from pathlib import Path
import sys

# Run from the repository root. In Colab, set this to the extracted project directory.
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'src' / 'forecast_holdout.py').exists():
    raise FileNotFoundError(f'Project root not found at {PROJECT_DIR}. Set PROJECT_DIR to the repository folder.')
sys.path.insert(0, str(PROJECT_DIR))
DATA_DIR = PROJECT_DIR / 'data' / 'raw'
OUTPUT_DIR = PROJECT_DIR / 'artifacts' / 'holdout_70_30'
MAX_SERIES = 0  # 0 = all series; set 1000 for a quick end-to-end run
BOOTSTRAP_REPLICATES = 1000
SEED = 42
print('Project:', PROJECT_DIR)
print('Data:', DATA_DIR)
print('Output:', OUTPUT_DIR)

In [ ]:
%pip install -r requirements.txt

In [ ]:
import hashlib
import json
import platform
from datetime import datetime, timezone

import lightgbm as lgb
import numpy as np
import pandas as pd

from src.forecast_experiments import CODES, HORIZON, SEED as MODEL_SEED, TRAIN_DAYS, fit, load_data, make_features, volatility
from src.forecast_holdout import (TRAIN_FRACTION, STATIC, MODELS, compact_feature_dtypes, model_parameters, rolling_holdout_forecasts, summarize_metrics, bootstrap_comparisons)
assert SEED == MODEL_SEED, 'Keep the reproducibility seed aligned with the training helpers.'
print({'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__, 'lightgbm': lgb.__version__})

## 1. Load and inspect the dataset

Sales days are kept in chronological order. Calendar and weekly prices are aligned to the sales history.

In [ ]:
required_files = ['sales_train_validation.csv', 'calendar.csv', 'sell_prices.csv']
missing = [name for name in required_files if not (DATA_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f'Missing {missing} under {DATA_DIR}. See README.md for data setup.')
meta, y, codes, days, calendar, sales_file, prices, price_file = load_data(DATA_DIR, MAX_SERIES)
if prices is None:
    raise ValueError('sell_prices.csv is required for this price-aware model.')
print(f'{len(meta):,} series x {len(days):,} daily observations')
print('Sales matrix:', y.shape, '| Price matrix:', prices.shape)
display(meta.head())

## 2. Split dates chronologically

The first 70% of dates are available for training. The final 30% stay held out. Boosting-round selection uses a validation slice wholly inside the training period, so test observations do not influence fitting or model selection.

In [ ]:
split_day = int(np.floor(y.shape[1] * TRAIN_FRACTION))
test_horizon = y.shape[1] - split_day
if split_day < max(TRAIN_DAYS, 52 * 7) + HORIZON or test_horizon < HORIZON:
    raise ValueError('Dataset is too short for the configured training and holdout windows.')
history = y[:, :split_day]
cutoff = split_day - 1
train_days = np.arange(28, split_day, dtype=np.int32)  # first 28 days are feature warm-up
print(f'Train: {days[0]} through {days[cutoff]} ({split_day:,} days, {TRAIN_FRACTION:.0%})')
print(f'Test:  {days[split_day]} through {days[-1]} ({test_horizon:,} days, {1-TRAIN_FRACTION:.0%})')
print(f'Eligible training labels: {len(meta) * len(train_days):,}')

## 3. Select training length and fit the final model

LightGBM early stopping selects the number of rounds on a training-period validation slice. A new model is then fitted on all eligible labeled dates in the 70% training partition.

In [ ]:
selector = fit(history, cutoff, calendar, codes, prices)
selected_rounds = max(1, int(selector.best_iteration or selector.current_iteration()))
del selector
train_features = compact_feature_dtypes(make_features(history, train_days, calendar, codes, prices))
train_labels = history[:, train_days].T.reshape(-1)
train_data = lgb.Dataset(train_features, label=train_labels, categorical_feature=CODES, free_raw_data=True)
model = lgb.train(model_parameters(), train_data, num_boost_round=selected_rounds)
del train_features, train_labels, train_data
print(f'Final model fitted with {selected_rounds} boosting rounds.')

## 4. Forecast the held-out dates

Predictions are recursive within each non-overlapping 28-day window. Actuals from completed earlier test windows become history at later origins; actuals inside the current forecast window are never used to build its predictions.

In [ ]:
predictions = rolling_holdout_forecasts(model, y, split_day, test_horizon, calendar, codes, prices)
actual = y[:, split_day:]
test_days = days[split_day:]
volatility_groups, zero_week_share = volatility(history)
print('Prediction arrays:', {name: values.shape for name, values in predictions.items()})

## 5. Evaluate and quantify uncertainty

Report pooled WAPE, MAE, and RMSE overall and by volatility group. A series-cluster bootstrap compares LightGBM with both baselines.

In [ ]:
metrics = summarize_metrics(actual, predictions, volatility_groups, test_days)
series_metrics = pd.DataFrame({
    'series_id': meta['id'].to_numpy(),
    'volatility_group': volatility_groups,
    'zero_week_share': zero_week_share,
    'actual_demand_sum': actual.sum(axis=1),
    **{f'absolute_error_{name}': np.abs(actual - forecast).sum(axis=1) for name, forecast in predictions.items()},
})
bootstrap = bootstrap_comparisons(series_metrics, BOOTSTRAP_REPLICATES)
display(metrics[metrics.volatility_group == 'all'][['model', 'WAPE', 'MAE', 'RMSE']].sort_values('WAPE'))
display(metrics[metrics.volatility_group != 'all'][['model', 'volatility_group', 'WAPE', 'MAE', 'RMSE']])
display(bootstrap)

## 6. Save model and reproducibility records

The output folder contains the LightGBM model, metadata, pooled and per-series metrics, bootstrap comparisons, and a run manifest with checksums and package versions.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
metrics.to_csv(OUTPUT_DIR / 'metrics.csv', index=False)
series_metrics.to_csv(OUTPUT_DIR / 'series_metrics.csv', index=False)
bootstrap.to_csv(OUTPUT_DIR / 'bootstrap_comparison.csv', index=False)
model_path = OUTPUT_DIR / 'lightgbm_price.txt'
model.save_model(str(model_path))
category_maps = {code: [str(value) for value in pd.Categorical(meta[source]).categories] for code, source in zip(CODES, STATIC)}
metadata = {
    'model': 'lightgbm_recursive_price', 'model_file': model_path.name,
    'feature_names': model.feature_name(), 'categorical_features': CODES,
    'category_code_maps': category_maps, 'horizon_days': HORIZON, 'history_days_required': 28,
    'selected_boosting_rounds': selected_rounds, 'training_fraction': TRAIN_FRACTION,
    'training_days': split_day, 'test_days': test_horizon, 'series_evaluated': len(meta),
    'seed': MODEL_SEED, 'sales_file_sha256': hashlib.sha256(sales_file.read_bytes()).hexdigest(),
    'sell_prices_file_sha256': hashlib.sha256(price_file.read_bytes()).hexdigest(),
    'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__, 'lightgbm': lgb.__version__,
}
manifest = {
    'evaluation': 'chronological 70/30 holdout', 'train_fraction': TRAIN_FRACTION,
    'test_fraction': round(1 - TRAIN_FRACTION, 2), 'forecast_window_days': HORIZON,
    'rolling_forecast_origins': int(np.ceil(test_horizon / HORIZON)),
    'train_start_day': days[0], 'train_end_day': days[cutoff],
    'test_start_day': days[split_day], 'test_end_day': days[-1],
    'training_days': split_day, 'test_days': test_horizon,
    'training_observations': int(len(meta) * len(train_days)), 'series_evaluated': len(meta),
    'selected_boosting_rounds': selected_rounds, 'bootstrap_replicates': BOOTSTRAP_REPLICATES,
    'seed': MODEL_SEED, 'sales_file_sha256': metadata['sales_file_sha256'],
    'sell_prices_file_sha256': metadata['sell_prices_file_sha256'],
    'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
    'lightgbm': lgb.__version__, 'run_timestamp_utc': datetime.now(timezone.utc).isoformat(),
}
(OUTPUT_DIR / 'model_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'run_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Saved model:', model_path)
print('Saved metrics:', OUTPUT_DIR / 'metrics.csv')

## Run instructions

From the repository root, open this notebook in Jupyter and run all cells. On Windows, create `.venv_cp312` with `py -3.12 -m venv .venv_cp312`, activate it, then install `requirements.txt`. In Colab, upload or extract the project and `data/raw` files, set `PROJECT_DIR`, then run all cells. Full-population training needs substantial RAM; use `MAX_SERIES = 1000` to check the complete pipeline first.